In [12]:
!pip install -q langchain-groq
%pip install -q langchain-community
%pip install -q langchain-classic

In [30]:
import os, requests
from google.colab import userdata
from langchain_groq import ChatGroq
from pathlib import Path
from langchain_core.prompts import (ChatPromptTemplate,
                                    HumanMessagePromptTemplate,
                                    SystemMessagePromptTemplate,
                                    AIMessagePromptTemplate,
                                    MessagesPlaceholder
                                    )
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_classic.chains.llm import LLMChain
from langchain_classic.memory import ConversationBufferMemory, ConversationBufferWindowMemory, ConversationSummaryMemory
from langchain_classic.globals import set_verbose


set_verbose(True)

In [14]:
def _read_env_file(name, filename="secrets.env"):
    """Look for KEY=value in secrets.env in the working dir or /content."""
    for folder in (Path.cwd(), Path("/content")):
        path = folder / filename
        if path.is_file():
            for line in path.read_text().splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, _, val = line.partition("=")
                if key.strip().removeprefix("export ").strip() == name:
                    return val.strip().strip('"').strip("'")
    return None

def get_secret(name, required=True):
    value = None
    try:
        value = userdata.get(name)      # works in the Colab web UI
    except Exception:
        pass                            # timeout / not found / no access
    if not value:
        value = _read_env_file(name)    # fallback: uploaded secrets.env
    if value:
        os.environ[name] = value
    elif required:
        raise RuntimeError(
            f"{name} not found in Colab Secrets or secrets.env. "
            "Upload secrets.env to the Colab server and re-run."
        )
    return value

groq_key = get_secret("GROQ_API_KEY")


print("GROQ key loaded:", bool(groq_key))


GROQ key loaded: True


In [15]:
llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    model_kwargs= {"seed" : 350 },
    temperature= 0,
    max_tokens= 250
)

chatmessage history

In [16]:
human_template = HumanMessagePromptTemplate.from_template('{question}')
system_template = SystemMessagePromptTemplate.from_template("you are a sarcastic chatbot that answers in sarcastic way")

chat_template = ChatPromptTemplate.from_messages( background_info.messages + [human_template])

print(chat_template.messages)

[HumanMessage(content='whats the biggest planet of our solar system ?', additional_kwargs={}, response_metadata={}), AIMessage(content='Jupiter. It’s more than twice as massive as all the other planets in the solar system combined', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})]


In [17]:
background_info = ChatMessageHistory()
background_info.add_user_message('whats the biggest planet of our solar system ?')
background_info.add_ai_message('Jupiter. It’s more than twice as massive as all the other planets in the solar system combined')

In [18]:
chain = LLMChain(
    llm = llm,
    prompt= chat_template
)

In [19]:
response =  chain.invoke({'question' : 'how wide it is than earth ?'})

print(response['text'])



> Entering new LLMChain chain...
Prompt after formatting:
Human: whats the biggest planet of our solar system ?
AI: Jupiter. It’s more than twice as massive as all the other planets in the solar system combined
Human: how wide it is than earth ?



> Finished chain.
**Jupiter vs. Earth – a size comparison**

| Property | Jupiter | Earth | How many × bigger |
|----------|---------|-------|-------------------|
| **Mean diameter** (equatorial) | **139 820 km** | 12 742 km | **≈ 11 times** |
| **Mean radius** | 69 911 km | 6 371 km | **≈ 11 times** |
| **Polar diameter** (because Jupiter is slightly flattened) | 133 708 km | — | — |
| **Volume** | 1 321 × 10¹⁵ km³ | 1.083 × 10¹² km³ | **≈ 1 300 times** |
| **Mass** | 1.


**Buffer Memory**

In [20]:
system_template = SystemMessagePromptTemplate.from_template("you are a sarcastic chatbot that answers in sarcastic way")
human_template = HumanMessagePromptTemplate.from_template(template = "{question}")
history_placeholder = MessagesPlaceholder(variable_name= "message_log")

In [21]:
chat_template = ChatPromptTemplate.from_messages(messages= [system_template,history_placeholder,human_template])

In [22]:
message_history = ChatMessageHistory()
message_history.add_user_message("whats the biggest planet of our solar system ?")
message_history.add_ai_message("Jupiter. Shocking, I know.")

In [23]:
memory = ConversationBufferMemory(
    memory_key= "message_log",
    chat_memory= message_history,
    return_messages= True
)

memory.load_memory_variables({})["message_log"]

/tmp/ipykernel_1498/560288507.py:1: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(


[HumanMessage(content='whats the biggest planet of our solar system ?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Jupiter. Shocking, I know.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [24]:
chain = LLMChain(
    llm = llm,
    memory = memory,
    prompt= chat_template
)

response = chain.invoke({"question": "and how far from earth ?"})

print(response["text"])



> Entering new LLMChain chain...
Prompt after formatting:
System: you are a sarcastic chatbot that answers in sarcastic way
Human: whats the biggest planet of our solar system ?
AI: Jupiter. Shocking, I know.
Human: and how far from earth ?

> Finished chain.
Oh, just a casual “hey‑you‑can‑see‑it‑in‑the‑night‑sky” kind of distance—roughly **588 million kilometers** (about 365 million miles) when Jupiter’s on the friendly side of the Sun, and up to **≈ 968 million kilometers** (≈ 602 million miles) when it decides to hide behind the Sun. In other words, it’s far enough that you’ll need a telescope, a spaceship, or a very patient imagination to get any closer. 🚀✨


In [25]:
window_memory = ConversationBufferWindowMemory(
    chat_memory= message_history,
    memory_key= "message_log",
    k = 2,
    return_messages= True
)

window_memory.load_memory_variables({})

/tmp/ipykernel_1498/188018495.py:1: LangChainDeprecationWarning: The class `ConversationBufferWindowMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  window_memory = ConversationBufferWindowMemory(


{'message_log': [HumanMessage(content='whats the biggest planet of our solar system ?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Jupiter. Shocking, I know.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='and how far from earth ?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Oh, just a casual “hey‑you‑can‑see‑it‑in‑the‑night‑sky” kind of distance—roughly **588\u202fmillion\u202fkilometers** (about 365\u202fmillion\u202fmiles) when Jupiter’s on the friendly side of the Sun, and up to **≈\u202f968\u202fmillion\u202fkilometers** (≈\u202f602\u202fmillion\u202fmiles) when it decides to hide behind the Sun. In other words, it’s far enough that you’ll need a telescope, a spaceship, or a very patient imagination to get any closer. 🚀✨', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}

In [26]:
chain = LLMChain(
    llm = llm,
    memory = window_memory,
    prompt = chat_template
)

response = chain.invoke({"question": "is it big ?"})
print(response["text"])



> Entering new LLMChain chain...
Prompt after formatting:
System: you are a sarcastic chatbot that answers in sarcastic way
Human: whats the biggest planet of our solar system ?
AI: Jupiter. Shocking, I know.
Human: and how far from earth ?
AI: Oh, just a casual “hey‑you‑can‑see‑it‑in‑the‑night‑sky” kind of distance—roughly **588 million kilometers** (about 365 million miles) when Jupiter’s on the friendly side of the Sun, and up to **≈ 968 million kilometers** (≈ 602 million miles) when it decides to hide behind the Sun. In other words, it’s far enough that you’ll need a telescope, a spaceship, or a very patient imagination to get any closer. 🚀✨
Human: is it big ?

> Finished chain.
Oh, absolutely tiny—just a **little** 11‑times‑Earth‑diameter, 318‑times‑Earth‑mass gas‑ball that barely nudges the Sun’s gravity. In other words, if you tried to fit it in a parking lot you’d need a parking lot the size of a small country. 🚗🪐


**Conversation Summary Memory**

In [81]:
template_sys = SystemMessagePromptTemplate.from_template(
   '''you are a tired chatbot that answers in a tired way
      current summary : {summary_key}
''')
template_h = HumanMessagePromptTemplate.from_template("{question}")

In [82]:
summary_chat_template = ChatPromptTemplate.from_messages([
    template_sys, 
    template_h
    ])

print(summary_chat_template.messages)

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['summary_key'], input_types={}, partial_variables={}, template='you are a tired chatbot that answers in a tired way\n      current summary : {summary_key}\n'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})]


In [83]:
summary_memory = ConversationSummaryMemory(
        memory_key= 'summary_key',
        llm= llm,
        return_messages= False
)
summary_memory.load_memory_variables({})

print(summary_memory)

llm=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.4', 'langchain': '1.4.8'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x7c9507ea3110>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7c9507ea3b10>, model_name='openai/gpt-oss-120b', temperature=1e-08, model_kwargs={'seed': 350}, groq_api_key=SecretStr('**********'), max_tokens=250) chat_memory=InMemoryChatMessageHistory(messages=[]) memory_key='summary_key'


In [86]:
chain = LLMChain(
    llm= llm,
    prompt= summary_chat_template,
    memory= summary_memory
)

#response = chain.invoke({'question': 'how many oceans are there in world ?'})
response = chain.invoke({'question': 'which is the biggest ?'})

print(response['text'])



> Entering new LLMChain chain...
Prompt after formatting:
System: you are a tired chatbot that answers in a tired way
      current summary : The human asks how many oceans there are in the world. The AI replies that there are five oceans—the Pacific, Atlantic, Indian, Southern (or Antarctic), and Arctic. The human then asks which is the biggest, and the AI answers that the Pacific Ocean is the biggest.

Human: which is the biggest ?


> Entering new LLMChain chain...
Prompt after formatting:
Progressively summarize the lines of conversation provided, adding onto the previous summary returning a new summary.

EXAMPLE
Current summary:
The human asks what the AI thinks of artificial intelligence. The AI thinks artificial intelligence is a force for good.

New lines of conversation:
Human: Why do you think artificial intelligence is a force for good?
AI: Because artificial intelligence will help humans reach their full potential.

New summary:
The human asks what the AI thinks of artifi